In [25]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import *
from sklearn.metrics import *
import dash
from dash import html, dcc, callback
from dash.dependencies import Input, Output
import plotly.express as px

In [3]:
#Read data from csv file
df = pd.read_csv('college.csv')
df.head()

,College.Name,Private,Apps,Accept,Enroll,Top10perc,Top25perc,F.Undergrad,P.Undergrad,Outstate,Room.Board,Books,Personal,PhD,Terminal,S.F.Ratio,perc.alumni,Expend,Grad.Rate
0,Abilene Christian University,Yes,1660,1232,721,23,52,2885,537,7440,3300,450,2200,70,78,18.1,12,7041,60
1,Adelphi University,Yes,2186,1924,512,16,29,2683,1227,12280,6450,750,1500,29,30,12.2,16,10527,56
2,Adrian College,Yes,1428,1097,336,22,50,1036,99,11250,3750,400,1165,53,66,12.9,30,8735,54
3,Agnes Scott College,Yes,417,349,137,60,89,510,63,12960,5450,450,875,92,97,7.7,37,19016,59
4,Alaska Pacific University,Yes,193,146,55,16,44,249,869,7560,4120,800,1500,76,72,11.9,2,10922,15


In [4]:
# Making two new features
df['Acc_rate'] = df['Accept'] / df['Apps']
df['Acc_offer'] = df['Enroll'] / df['Accept']

In [9]:
# At this point I'll drop Rutgers at New Brunswick because it has 40000 Apps
df.drop(index = 483, inplace = True)

In [10]:
dummy_var = pd.get_dummies(df['Private'])
dummy_var.drop(columns = 'No', inplace = True)
dummy_var

,Yes
0,True
1,True
2,True
3,True
4,True
...,...
772,False
773,True
774,True
775,True


In [11]:
df['Private'] = dummy_var
df

,College.Name,Private,Apps,Accept,Enroll,Top10perc,Top25perc,F.Undergrad,P.Undergrad,Outstate,...,Books,Personal,PhD,Terminal,S.F.Ratio,perc.alumni,Expend,Grad.Rate,Acc_rate,Acc_offer
0,Abilene Christian University,True,1660,1232,721,23,52,2885,537,7440,...,450,2200,70,78,18.1,12,7041,60,0.742169,0.585227
1,Adelphi University,True,2186,1924,512,16,29,2683,1227,12280,...,750,1500,29,30,12.2,16,10527,56,0.880146,0.266112
2,Adrian College,True,1428,1097,336,22,50,1036,99,11250,...,400,1165,53,66,12.9,30,8735,54,0.768207,0.306290
3,Agnes Scott College,True,417,349,137,60,89,510,63,12960,...,450,875,92,97,7.7,37,19016,59,0.836930,0.392550
4,Alaska Pacific University,True,193,146,55,16,44,249,869,7560,...,800,1500,76,72,11.9,2,10922,15,0.756477,0.376712
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
772,Worcester State College,False,2197,1515,543,4,26,3089,2029,6797,...,500,1200,60,60,21.0,14,4469,40,0.689577,0.358416
773,Xavier University,True,1959,1805,695,24,47,2849,1107,11520,...,600,1250,73,75,13.3,31,9189,83,0.921388,0.385042
774,Xavier University of Louisiana,True,2097,1915,695,34,61,2793,166,6900,...,617,781,67,75,14.4,20,8323,49,0.913209,0.362924
775,Yale University,True,10705,2453,1317,95,99,5217,83,19840,...,630,2115,96,96,5.8,49,40386,99,0.229145,0.536894


In [17]:
df.drop(columns = 'College.Name', inplace = True)

In [18]:
#Split data into test and train
train, test = train_test_split(df, train_size = 0.7, random_state = 1234)
print(train.shape)
print(test.shape)

(543, 20)
(233, 20)


In [19]:
X_train = train.drop(columns = 'Apps')
y_train = train['Apps']
X_test = test.drop(columns = 'Apps')
y_test = test['Apps']
print(X_train.shape)
print(X_test.shape)

(543, 19)
(233, 19)


In [20]:
from sklearn.neighbors import KNeighborsRegressor
model_3 = KNeighborsRegressor(n_neighbors = 5)
model_3.fit(X_train, y_train)

KNeighborsRegressor()

In [23]:
## Predict response variable in the train dataset
y_train_pred_3 = model_3.predict(X_train)

#Calculare residuals
res_3 = y_train - y_train_pred_3

#The root mean squared error
print('RMSE: {:0.3f}'.format(root_mean_squared_error(y_train, y_train_pred_3)))
#The root mean squared error
print('MAPE: {:0.3f}'.format(mean_absolute_percentage_error(y_train, y_train_pred_3) * 100))
#The coefficient of determination
print('R2: {:0.3f}'.format(r2_score(y_train, y_train_pred_3) * 100))

RMSE: 971.287
MAPE: 29.987
R2: 92.428


In [28]:
#Version 000
app000 = dash.Dash()
app000.layout = html.Div(children = [html.H1('Data Science Course Project'), 
                                     html.H2('kNN Regression on Synthetic Data')])
app000.run(jupyter_mode = 'tab', port=8051)

Dash app running on http://127.0.0.1:8051/


<IPython.core.display.Javascript object>

In [47]:
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.express as px
from sklearn.neighbors import KNeighborsRegressor
import numpy as np
import pandas as pd

# Assume these are preloaded
# X_train: DataFrame with 17 features
# y_train: Series or 1D array of target values

# Dash App
app001 = dash.Dash()

app001.layout = html.Div(children=[
    html.H1('Sample Dashboard'),
    html.H2('kNN Regression on Real Data'),
    dcc.Dropdown(
        options=[{'label': str(k), 'value': k} for k in [2, 5, 10, 15, 20, 25, 30]],
        value=10,
        id='kNN',
        style={'width': '30%'}
    ),
    dcc.Graph(id='scatter plot')
])

@app001.callback(
    Output('scatter plot', 'figure'),
    Input('kNN', 'value')
)
def update_graph(k):
    model = KNeighborsRegressor(n_neighbors=k, weights='uniform', algorithm='brute')
    model.fit(X_train, y_train)

    y_pred = model.predict(X_train)

    fig = px.scatter(x=y_pred, y=y_train,
                     labels={'x': 'Predicted', 'y': 'Actual'},
                     title=f'kNN Regression (k = {k})')

    fig.add_scatter(x=y_pred, y=y_pred, mode='lines', name='Ideal Fit (y = x)',
                    line=dict(color='red', dash='dash'))

    fig.update_layout(xaxis_title='Predicted Value', yaxis_title='Actual Value')

    return fig

app001.run(jupyter_mode='tab', port=8051)

Dash app running on http://127.0.0.1:8051/


<IPython.core.display.Javascript object>

In [49]:
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.express as px
from sklearn.neighbors import KNeighborsRegressor

# Make sure X_train and y_train are already defined:
# X_train: DataFrame with 17 features (including 'PhD')
# y_train: Series or array

app001 = dash.Dash()

app001.layout = html.Div(children=[
    html.H1('Sample Dashboard'),
    html.H2('kNN Regression on Real Data'),
    dcc.Dropdown(
        options=[{'label': str(k), 'value': k} for k in [2, 5, 10, 15, 20, 25, 30]],
        value=10,
        id='kNN',
        style={'width': '30%'}
    ),
    dcc.Graph(id='scatter plot')
])

@app001.callback(
    Output('scatter plot', 'figure'),
    Input('kNN', 'value')
)
def update_graph(k):
    model = KNeighborsRegressor(n_neighbors=k, weights='uniform', algorithm='brute')
    model.fit(X_train, y_train)

    y_pred = model.predict(X_train)

    fig = px.scatter(
        x=X_train['PhD'],
        y=y_train,
        labels={'x': 'PhD', 'y': 'Actual Target'},
        title=f'kNN Regression: Actual vs Predicted (k = {k})'
    )

    # Add predicted values as a second trace
    fig.add_scatter(
        x=X_train['PhD'],
        y=y_pred,
        mode='markers',
        name='Predicted',
        marker=dict(color='red', symbol='x')
    )

    fig.update_layout(
        xaxis_title='PhD Feature',
        yaxis_title='Target Value'
    )

    return fig

app001.run(jupyter_mode='tab', port=8051)

Dash app running on http://127.0.0.1:8051/


<IPython.core.display.Javascript object>

In [50]:
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.express as px
from sklearn.neighbors import KNeighborsRegressor

# Assume X_train and y_train are preloaded
# X_train is a DataFrame with 17 features
# y_train is a Series or 1D array

feature_names = X_train.columns.tolist()  # Get all 17 feature names

app001 = dash.Dash()

app001.layout = html.Div(children=[
    html.H1('Sample Dashboard'),
    html.H2('kNN Regression on Real Data'),

    html.Div([
        html.Label("Choose number of neighbors (k):"),
        dcc.Dropdown(
            options=[{'label': str(k), 'value': k} for k in [2, 5, 10, 15, 20, 25, 30]],
            value=10,
            id='kNN',
            style={'width': '40%'}
        ),
    ], style={'padding': '10px'}),

    html.Div([
        html.Label("Choose feature to visualize:"),
        dcc.Dropdown(
            options=[{'label': feature, 'value': feature} for feature in feature_names],
            value='PhD',  # Default
            id='feature',
            style={'width': '40%'}
        ),
    ], style={'padding': '10px'}),

    dcc.Graph(id='scatter plot')
])

@app001.callback(
    Output('scatter plot', 'figure'),
    Input('kNN', 'value'),
    Input('feature', 'value')
)
def update_graph(k, selected_feature):
    model = KNeighborsRegressor(n_neighbors=k, weights='uniform', algorithm='brute')
    model.fit(X_train, y_train)

    y_pred = model.predict(X_train)

    fig = px.scatter(
        x=X_train[selected_feature],
        y=y_train,
        labels={'x': selected_feature, 'y': 'Actual Target'},
        title=f'kNN Regression: Actual vs Predicted (k = {k})'
    )

    # Add predicted values as a second trace
    fig.add_scatter(
        x=X_train[selected_feature],
        y=y_pred,
        mode='markers',
        name='Predicted',
        marker=dict(color='red', symbol='x')
    )

    fig.update_layout(
        xaxis_title=selected_feature,
        yaxis_title='Target Value'
    )

    return fig

app001.run(jupyter_mode='tab', port=8051)


Dash app running on http://127.0.0.1:8051/


<IPython.core.display.Javascript object>

In [51]:
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.express as px
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_percentage_error
import numpy as np

# Assume X_train and y_train are preloaded
# X_train: DataFrame with 17 features (including 'PhD')
# y_train: Series or array

feature_names = X_train.columns.tolist()

app001 = dash.Dash()

app001.layout = html.Div(children=[
    html.H1('kNN Regression Dashboard'),
    html.H2('Explore Model Performance with Adjustable k and Feature'),

    html.Div([
        html.Label("Choose number of neighbors (k):"),
        dcc.Dropdown(
            options=[{'label': str(k), 'value': k} for k in [2, 5, 10, 15, 20, 25, 30]],
            value=10,
            id='kNN',
            style={'width': '40%'}
        ),
    ], style={'padding': '10px'}),

    html.Div([
        html.Label("Choose feature to visualize:"),
        dcc.Dropdown(
            options=[{'label': feature, 'value': feature} for feature in feature_names],
            value='PhD',
            id='feature',
            style={'width': '40%'}
        ),
    ], style={'padding': '10px'}),

    dcc.Graph(id='scatter plot')
])

@app001.callback(
    Output('scatter plot', 'figure'),
    Input('kNN', 'value'),
    Input('feature', 'value')
)
def update_graph(k, selected_feature):
    model = KNeighborsRegressor(n_neighbors=k, weights='uniform', algorithm='brute')
    model.fit(X_train, y_train)

    y_pred = model.predict(X_train)

    # Calculate metrics
    r2 = r2_score(y_train, y_pred)
    rmse = mean_squared_error(y_train, y_pred, squared=False)
    mape = mean_absolute_percentage_error(y_train, y_pred) * 100  # percentage

    # Format metrics for title
    metrics_text = f"R² = {r2:.3f} | RMSE = {rmse:.3f} | MAPE = {mape:.2f}%"

    # Create plot
    fig = px.scatter(
        x=X_train[selected_feature],
        y=y_train,
        labels={'x': selected_feature, 'y': 'Actual Target'},
        title=f'kNN Regression (k = {k}) | {metrics_text}'
    )

    # Add prediction scatter
    fig.add_scatter(
        x=X_train[selected_feature],
        y=y_pred,
        mode='markers',
        name='Predicted',
        marker=dict(color='red', symbol='x')
    )

    fig.update_layout(
        xaxis_title=selected_feature,
        yaxis_title='Target Value',
        legend=dict(x=0.01, y=0.99)
    )

    return fig

app001.run(jupyter_mode='tab', port=8051)

Dash app running on http://127.0.0.1:8051/


<IPython.core.display.Javascript object>

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:483: FutureWarning:

'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:483: FutureWarning:

'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.



In [57]:
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.express as px
import numpy as np
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_percentage_error

# Assume X_train, y_train defined

feature_names = X_train.columns.tolist()

app = dash.Dash()

app.layout = html.Div([
    html.H1('ML Regression Models Dashboard'),

    html.Div([
        html.Label("Choose regression model:"),
        dcc.Dropdown(
            options=[
                {'label': 'k-Nearest Neighbors', 'value': 'knn'},
                {'label': 'Linear Regression', 'value': 'linear'},
                {'label': 'Decision Tree', 'value': 'tree'},
                {'label': 'Random Forest', 'value': 'forest'},
                {'label': 'Support Vector Regressor', 'value': 'svr'},
                {'label': 'Gradient Boosting', 'value': 'gbr'}
            ],
            value='knn',
            id='model',
            style={'width': '40%'}
        ),
    ], style={'padding': '10px'}),

    html.Div([
        html.Label("Choose number of neighbors (k):"),
        dcc.Dropdown(
            options=[{'label': str(k), 'value': k} for k in [2, 5, 10, 15, 20, 25, 30]],
            value=10,
            id='kNN',
            style={'width': '40%'}
        ),
    ], style={'padding': '10px'}),

    html.Div([
        html.Label("Choose feature to visualize:"),
        dcc.Dropdown(
            options=[{'label': f, 'value': f} for f in feature_names],
            value='PhD',
            id='feature',
            style={'width': '40%'}
        ),
    ], style={'padding': '10px'}),

    dcc.Graph(id='scatter plot')
])

@app.callback(
    Output('scatter plot', 'figure'),
    Output('kNN', 'disabled'),
    Input('model', 'value'),
    Input('kNN', 'value'),
    Input('feature', 'value')
)
def update_graph(model_name, k, selected_feature):
    # Choose model
    if model_name == 'knn':
        model = KNeighborsRegressor(n_neighbors=k, weights='uniform', algorithm='brute')
        k_disabled = False
    elif model_name == 'linear':
        model = LinearRegression()
        k_disabled = True
    elif model_name == 'tree':
        model = DecisionTreeRegressor(random_state=42)
        k_disabled = True
    elif model_name == 'forest':
        model = RandomForestRegressor(n_estimators=100, random_state=42)
        k_disabled = True
    elif model_name == 'svr':
        model = SVR()
        k_disabled = True
    elif model_name == 'gbr':
        model = GradientBoostingRegressor(random_state=42)
        k_disabled = True
    else:
        model = LinearRegression()
        k_disabled = True

    model.fit(X_train, y_train)
    y_pred = model.predict(X_train)

    r2 = r2_score(y_train, y_pred)
    rmse = mean_squared_error(y_train, y_pred, squared=False)
    mape = mean_absolute_percentage_error(y_train, y_pred) * 100

    metrics_text = f"R² = {r2:.3f} | RMSE = {rmse:.3f} | MAPE = {mape:.2f}%"

    fig = px.scatter(
        x=X_train[selected_feature],
        y=y_train,
        labels={'x': selected_feature, 'y': 'Actual Target'},
        title=f"{model_name.upper()} Regression (k={k if model_name=='knn' else '-'}) | {metrics_text}"
    )

    fig.add_scatter(
        x=X_train[selected_feature],
        y=y_pred,
        mode='markers',
        name='Predicted',
        marker=dict(color='red', symbol='x')
    )

    fig.update_layout(
        xaxis_title=selected_feature,
        yaxis_title='Target Value',
        legend=dict(x=0.01, y=0.99)
    )

    return fig, k_disabled

app.run(jupyter_mode='tab', port=8051)

Dash app running on http://127.0.0.1:8051/


<IPython.core.display.Javascript object>

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:483: FutureWarning:

'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.



In [66]:
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.express as px
import plotly.graph_objs as go
import numpy as np
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_percentage_error

# === Make sure X_train, y_train, feature_names are defined in your environment ===
# For example:
# import pandas as pd
# X_train = pd.read_csv('X_train.csv')  # DataFrame with 17 features including 'PhD'
# y_train = pd.read_csv('y_train.csv')['target']
# feature_names = X_train.columns.tolist()

feature_names = X_train.columns.tolist()

app = dash.Dash(__name__)

app.layout = html.Div([
    html.H1('ML Regression Models Dashboard'),

    html.Div([
        html.Label("Choose regression model:"),
        dcc.Dropdown(
            id='model',
            options=[
                {'label': 'k-Nearest Neighbors', 'value': 'knn'},
                {'label': 'Linear Regression', 'value': 'linear'},
                {'label': 'Decision Tree', 'value': 'tree'},
                {'label': 'Random Forest', 'value': 'forest'},
                #{'label': 'Support Vector Regressor', 'value': 'svr'},  # Removed
                {'label': 'Gradient Boosting', 'value': 'gbr'}
            ],
            value='knn',
            style={'width': '50%'}
        ),
    ], style={'padding': '10px'}),

    # Hyperparameter inputs, all present but shown/hidden per model
    html.Div([
        html.Div([
            html.Label("Number of neighbors (k):"),
            dcc.Dropdown(
                id='kNN',
                options=[{'label': str(k), 'value': k} for k in [2, 5, 10, 15, 20, 25, 30]],
                value=10,
                style={'width': '50%'}
            )
        ], id='knn-div'),

        html.Div([
            html.Label("Max Depth:"),
            dcc.Input(id='tree-max-depth', type='number', min=1, max=50, step=1, value=5, style={'width': '30%'})
        ], id='tree-div'),

        html.Div([
            html.Label("Number of Trees (n_estimators):"),
            dcc.Input(id='forest-n-estimators', type='number', min=10, max=500, step=10, value=100, style={'width': '30%'}),
            html.Br(),
            html.Label("Max Depth:"),
            dcc.Input(id='forest-max-depth', type='number', min=1, max=50, step=1, value='', placeholder='None', style={'width': '30%'})
        ], id='forest-div'),

        # Removed SVR hyperparameter block completely (svr-div)

        html.Div([
            html.Label("Number of Trees (n_estimators):"),
            dcc.Input(id='gbr-n-estimators', type='number', min=10, max=500, step=10, value=100, style={'width': '30%'}),
            html.Br(),
            html.Label("Max Depth:"),
            dcc.Input(id='gbr-max-depth', type='number', min=1, max=50, step=1, value=3, style={'width': '30%'})
        ], id='gbr-div'),
    ]),

    html.Div([
        html.Label("Choose feature to visualize:"),
        dcc.Dropdown(
            id='feature',
            options=[{'label': f, 'value': f} for f in feature_names],
            value='PhD',
            style={'width': '50%'}
        ),
    ], style={'padding': '10px'}),

    dcc.Graph(id='scatter plot')
])


@app.callback(
    Output('knn-div', 'style'),
    Output('tree-div', 'style'),
    Output('forest-div', 'style'),
    Output('gbr-div', 'style'),
    Input('model', 'value')
)
def show_hide_hyperparameters(model_name):
    hide = {'display': 'none'}
    show = {'display': 'block', 'padding': '10px'}

    return (
        show if model_name == 'knn' else hide,
        show if model_name == 'tree' else hide,
        show if model_name == 'forest' else hide,
        show if model_name == 'gbr' else hide,
    )


@app.callback(
    Output('scatter plot', 'figure'),
    Input('model', 'value'),
    Input('feature', 'value'),
    Input('kNN', 'value'),
    Input('tree-max-depth', 'value'),
    Input('forest-n-estimators', 'value'),
    Input('forest-max-depth', 'value'),
    Input('gbr-n-estimators', 'value'),
    Input('gbr-max-depth', 'value'),
)
def update_graph(model_name, selected_feature,
                 k, tree_max_depth, forest_n_estimators, forest_max_depth,
                 gbr_n_estimators, gbr_max_depth):

    import pandas as pd

    if selected_feature not in X_train.columns:
        fig = go.Figure()
        fig.add_annotation(text=f"Feature '{selected_feature}' not found in data.",
                           xref="paper", yref="paper",
                           showarrow=False, font=dict(size=20))
        return fig

    X_feature = X_train[selected_feature]
    y = y_train

    valid_idx = X_feature.dropna().index.intersection(y.dropna().index)
    X_feature = X_feature.loc[valid_idx]
    y = y.loc[valid_idx]

    # Fill or cast hyperparameters:
    if k is None: k = 10
    if tree_max_depth is None: tree_max_depth = 5
    if forest_n_estimators is None: forest_n_estimators = 100
    if forest_max_depth == '' or forest_max_depth is None: forest_max_depth = None
    else: forest_max_depth = int(forest_max_depth)
    if gbr_n_estimators is None: gbr_n_estimators = 100
    if gbr_max_depth is None: gbr_max_depth = 3

    # Select model
    if model_name == 'knn':
        model = KNeighborsRegressor(n_neighbors=k, weights='uniform', algorithm='brute')
    elif model_name == 'linear':
        model = LinearRegression()
    elif model_name == 'tree':
        model = DecisionTreeRegressor(max_depth=tree_max_depth, random_state=42)
    elif model_name == 'forest':
        model = RandomForestRegressor(n_estimators=forest_n_estimators,
                                      max_depth=forest_max_depth,
                                      random_state=42)
    elif model_name == 'gbr':
        model = GradientBoostingRegressor(n_estimators=gbr_n_estimators,
                                          max_depth=gbr_max_depth,
                                          random_state=42)
    else:
        model = LinearRegression()

    model.fit(X_feature.values.reshape(-1, 1), y)
    y_pred = model.predict(X_feature.values.reshape(-1, 1))

    r2 = r2_score(y, y_pred)
    rmse = mean_squared_error(y, y_pred, squared=False)
    mape = mean_absolute_percentage_error(y, y_pred) * 100

    metrics_text = f"R² = {r2:.3f} | RMSE = {rmse:.3f} | MAPE = {mape:.2f}%"

    fig = px.scatter(
        x=X_feature,
        y=y,
        labels={'x': selected_feature, 'y': 'Actual Target'},
        title=f"{model_name.upper()} Regression | {metrics_text}"
    )

    fig.add_scatter(
        x=X_feature,
        y=y_pred,
        mode='markers',
        name='Predicted',
        marker=dict(color='red', symbol='x')
    )

    fig.update_layout(
        xaxis_title=selected_feature,
        yaxis_title='Target Value',
        legend=dict(x=0.01, y=0.99)
    )

    return fig


app.run(jupyter_mode='tab', port=8051)

Dash app running on http://127.0.0.1:8051/


<IPython.core.display.Javascript object>

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:483: FutureWarning:

'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:483: FutureWarning:

'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:483: FutureWarning:

'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:483: FutureWarning:

'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.

C:\Progr

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:483: FutureWarning:

'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:483: FutureWarning:

'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:483: FutureWarning:

'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:483: FutureWarning:

'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.

C:\Progr